<a href="https://colab.research.google.com/github/reynaudnangue28/test/blob/main/CO2_Emission_Project_Optminization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Optimization regression models (Optimization with GridSearchCV , RandomizedSearchCV and BayesSearchCV)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from skopt import BayesSearchCV
from scipy.stats import loguniform
from skopt.space import Real, Categorical, Integer
from sklearn.model_selection import GridSearchCV


# Google Colab to access and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ---  load a mapping csv---
try:
    mapping_df = pd.read_csv('/content/drive/MyDrive/Data/EEA2024CO2Data/eea_column_mapping_with_nulls.csv')
    cols_to_keep = mapping_df[mapping_df['keep'] == True]
    rename_map = dict(zip(cols_to_keep['column_name_original'], cols_to_keep['column_name_new']))
except FileNotFoundError:
    print("Error : file is missing.")
    exit()

# ---  load the EEA file  ---

df1 = pd.read_csv(
    "/content/drive/MyDrive/Data/EEA2024CO2Data/data.csv",
    nrows=7000000,
    low_memory=False
)

#Reading Between Raws 7 million and 10.8 million

df2 = pd.read_csv(
    "/content/drive/MyDrive/Data/EEA2024CO2Data/data.csv",
    skiprows=range(1, 7_000_001),  # skip header + first 7M rows
    low_memory=False
)

# ---  Filter and rename ---
# choose only the columns with 'keep=True'
df1 = df1[rename_map.keys()]
# Rename the columns
df1 = df1.rename(columns=rename_map)

# choose only the columns with 'keep=True'
df2 = df2[rename_map.keys()]
# Rename the columns
df2 = df2.rename(columns=rename_map)

#Concat of df1 and df2
df = pd.concat([df1, df2], axis=0, ignore_index=True)

# ---  end ---

fuel_map   = {
    'petrol': "Petrol",
    'diesel': "Diesel",
    'petrol/electric': "Petrol-Electric Hybrid",
    'diesel/electric': "Diesel-Electric Hybrid",
    'electric': "100% Electric",
    'lpg': "LPG",
    'ng': "Natural Gas",
    'e85': "Ethanol 85%",
    'hydrogen': "Hydrogen",
    'unknown': "Unknown",
    'es': "Petrol"
}

df['fuel_type'] = df['fuel_type'].str.strip().str.lower().replace(fuel_map)

#  Basic Preprocessing

#numeric features
features = [
    "mass_kg",
    "wltp_mass_kg",
    "engine_cc",
    "power_kw",
    "electric_consumption",
    "fuel_type"
]
target = 'co2_wltp'

df['engine_cc'] = df['engine_cc'].fillna(0)


df['electric_consumption'] = df['electric_consumption'].fillna(0)

# Drop rows with missing values in these specific columns
df = df.dropna(subset=features + [target])
X = pd.get_dummies(df[features], columns=['fuel_type'], drop_first=True)
y = df[target]


#  Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#  Evaluation Function
def evaluate_model(model, X_test, y_test, name):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print(f"--- {name} Performance ---")
    print(f"MAE (Mean Absolute Error): {mae:.2f} g/km")
    print(f"RMSE (Root Mean Squared Error): {rmse:.2f} g/km")
    print(f"R² score(Coefficient of Determination): {r2:.4f}")

#  Optimization with GridSearchCV , RandomizedSearchCV and BayesSearchCV

X_train_sub = X_train.sample(n=100000, random_state=42)
y_train_sub = y_train.loc[X_train_sub.index]

# Define the parameters
# We focus on depth and split points to prevent overfitting on the EEA dataset
rf_param = {
    'n_estimators': [100],
    'max_depth': [15,20],
    'min_samples_split': [5,10],
    'max_features': ['sqrt']
}

et_param = {
    'n_estimators': [100, 200],
    'max_depth': [15, 20],
    'bootstrap': [True],
    'max_features': ['sqrt']
}
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(max_iter=100, random_state=42))
])

#  Setup GridSearchCV
# cv=3 provides a good balance between speed and validation accuracy
rf_grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_param,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
et_grid_search = GridSearchCV(
    estimator=ExtraTreesRegressor(random_state=42),
    param_grid=et_param,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
    )
nn_param_grid = {
    'mlp__hidden_layer_sizes': [(50,), (100, 50)],
    'mlp__activation': ['relu', 'tanh'],
    'mlp__alpha': [0.0001, 0.001]
}
nn_grid_search = GridSearchCV(
    mlp_pipeline, nn_param_grid,
    cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
    )
# Setup RandomizedSearchCV
rf_random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=rf_param,
    n_iter=50,  # Number of random combinations to try
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)
et_random_search = RandomizedSearchCV(
    estimator=ExtraTreesRegressor(random_state=42),
    param_distributions=et_param,
    n_iter=50,  # Number of random combinations to try
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)
nn_param_dist = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (100, 50), (128, 64, 32)],
    'mlp__alpha': loguniform(1e-5, 1e-2),
    'mlp__learning_rate_init': [0.001, 0.01]
}
nn_random_search = RandomizedSearchCV(
    mlp_pipeline, nn_param_dist, n_iter=10, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1
    )
# Setup BayesSearchCV
rf_bayes_search = BayesSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    search_spaces={
    'n_estimators': [100],
    'max_depth': [15,20],
    'min_samples_split': [5,10],
    'max_features': ['sqrt']
    },
    n_iter=32,                              # Smart iterations
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)
et_bayes_search = BayesSearchCV(
    estimator=ExtraTreesRegressor(random_state=42),
    search_spaces={
    'n_estimators': [100],
    'max_depth': [15,20],
    'min_samples_split': [5,10],
    'max_features': ['sqrt']
    },
    n_iter=32,                              # Smart iterations
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)
search_space = {
    'mlp__hidden_layer_sizes': Categorical([(100,), (100, 50), (128, 64, 32)]),
    'mlp__alpha': Real(1e-5, 1e-2, prior='log-uniform'),
    'mlp__activation': Categorical(['relu', 'tanh'])
}
nn_bayes_search = BayesSearchCV(
     mlp_pipeline, search_space, n_iter=15, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1
     )
#  Fit the model to find the best combination
rf_grid_search.fit(X_train_sub, y_train_sub)
et_grid_search.fit(X_train_sub, y_train_sub)
nn_grid_search.fit(X_train_sub, y_train_sub)
rf_random_search.fit(X_train_sub, y_train_sub)
et_random_search.fit(X_train_sub, y_train_sub)
nn_random_search.fit(X_train_sub, y_train_sub)
rf_bayes_search.fit(X_train_sub, y_train_sub)
et_bayes_search.fit(X_train_sub, y_train_sub)
nn_bayes_search.fit(X_train_sub, y_train_sub)

# Convert to float32 to save 50% memory before full training
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

#  Extract results
print("Train on the full data...")
best_grid_rf = rf_grid_search.best_estimator_.fit(X_train, y_train)
best_grid_et = et_grid_search.best_estimator_.fit(X_train, y_train)
best_grid_nn = nn_grid_search.best_estimator_.fit(X_train, y_train)
best_random_rf = rf_random_search.best_estimator_.fit(X_train, y_train)
best_random_et = et_random_search.best_estimator_.fit(X_train, y_train)
best_random_nn = nn_random_search.best_estimator_.fit(X_train, y_train)
best_bayes_rf = rf_bayes_search.best_estimator_.fit(X_train, y_train)
best_bayes_et = et_bayes_search.best_estimator_.fit(X_train, y_train)
best_bayes_nn = nn_bayes_search.best_estimator_.fit(X_train, y_train)

# Final Evaluation
print("Optimization of Random Forest with GridSearchCV")
evaluate_model(best_grid_rf, X_test, y_test, "Optimized Random Forest")
evaluate_model(best_grid_et, X_test, y_test, "Optimized Extra Trees")
evaluate_model(best_grid_nn, X_test, y_test, "Optimized Neural Networks")
print("Optimization of Random Forest with RandomSearchCV")
evaluate_model(best_random_rf, X_test, y_test, "Optimized Random Forest")
evaluate_model(best_random_et, X_test, y_test, "Optimized Extra Trees")
evaluate_model(best_random_nn, X_test, y_test, "Optimized Neural Networks")
print("Optimization of Random Forest with BayesSearchCV")
evaluate_model(best_bayes_rf, X_test, y_test, "Optimized Random Forest")
evaluate_model(best_bayes_et, X_test, y_test, "Optimized Extra Trees")
evaluate_model(best_bayes_nn, X_test, y_test, "Optimized Neural Networks")

# Display the most important features
def plot_importance(model, name):
    feat_importances = pd.Series(model.feature_importances_, index=X.columns)
    plt.figure(figsize=(10,6))
    feat_importances.nlargest(15).plot(kind='barh')
    plt.title(f"Top 15 Features - {name}")
    plt.show()

plot_importance(best_grid_rf, "Random Forest with GridSearchCV")
plot_importance(best_grid_et, "Extra Trees with GridSearchCV")
plot_importance(best_grid_nn, "Neural Networks with GridSearchCV")
plot_importance(best_random_rf, "Random Forest with RandomSearchCV")
plot_importance(best_random_et, "Extra Trees with RandomSearchCV")
plot_importance(best_random_nn, "Neural Networks with RandomSearchCV")
plot_importance(best_bayes_rf, "Random Forest with BayesSearchCV")
plot_importance(best_bayes_et, "Extra Trees with BayesSearchCV")
plot_importance(best_bayes_nn, "Neural Networks with BayesSearchCV")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fitting 3 folds for each of 4 candidates, totalling 12 fits


/usr/local/lib/python3.12/dist-packages/skopt/space/space.py:116: UserWarning: Dimension [15, 20] was inferred to Integer(low=15, high=20, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(15, 20), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/space/space.py:116: UserWarning: Dimension [5, 10] was inferred to Integer(low=5, high=10, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred to Categorical(categories=(5, 10), prior=None). See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skopt/space/space.py:116: UserWarning: Dimension [15, 20] was inferred to Integer(low=15, high=20, prior='uniform', transform='identity'). In upcoming versions of scikit-optimize, it will be inferred

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Fitting 3 folds for each of 4 candidates, totalling 12 fits


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=50. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=50. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 4 candidates, totalling 12 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


...

/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(20), np.str_('sqrt'), np.int64(5), np.int64(100)] before, using random point [np.int64(17), 'sqrt', np.int64(6), 100]
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(20), np.str_('sqrt'), np.int64(5), np.int64(100)] before, using random point [np.int64(16), 'sqrt', np.int64(7), 100]
  warnings.warn(


Fitting 3 folds for each of 1 candidates, totalling 3 fits


ValueError: Not all points are within the bounds of the space.

# Neuer Abschnitt

In [ ]:
!pip install opencv-python
!pip install scikit-optimize


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
